# Lesson 6.6: The Two-Sample t-Test

Compare exam scores of 30 students who studied with music and 30 who studied in silence, with Welch's and the pooled two-sample t-test.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the two groups

The scores below are built to match the lesson: music averages 74 (SD 8), silence averages 79 (SD 7), 30 students each. Different students are in each group, so the groups are independent.

In [1]:
import numpy as np
from scipy import stats

music = np.array([77.91, 82.44, 77.77, 62.24, 83.23, 78.87, 69.52, 80.15, 78.09, 77.42, 74.9, 79.82, 67.63, 73.08, 70.05, 80.32, 75.01, 71.85, 67.2, 72.18, 74.71, 72.01, 86.93, 84.2, 48.86, 56.67, 72.97, 70.62, 76.66, 76.69])
silence = np.array([72.97, 64.46, 82.5, 83.47, 76.26, 80.89, 90.07, 85.63, 83.21, 83.08, 72.47, 84.97, 67.1, 76.11, 75.39, 66.56, 77.62, 89.61, 76.05, 70.12, 77.75, 84.22, 79.7, 78.36, 68.74, 81.8, 81.7, 87.55, 88.56, 83.08])

for name, g in [('music', music), ('silence', silence)]:
    print(f'{name}: n = {len(g)}, mean = {g.mean():.2f}, sd = {g.std(ddof=1):.2f}')

music: n = 30, mean = 74.00, sd = 8.00
silence: n = 30, mean = 79.00, sd = 7.00


## 2. Welch's t-test (the safe default)

equal_var=False gives Welch's test, which does not assume the two groups have the same spread. The degrees of freedom are not a whole number because they are adjusted for the unequal spreads.

In [2]:
welch = stats.ttest_ind(music, silence, equal_var=False)

print(f't  = {welch.statistic:.3f}')
print(f'df = {welch.df:.1f}')
print(f'p  = {welch.pvalue:.4f}')

t  = -2.576
df = 57.0
p  = 0.0126


## 3. The pooled t-test

equal_var=True blends the two variances into one. Here the group sizes are equal and the spreads are similar, so the two tests agree closely. They would not agree if the smaller group were also the noisier one.

In [3]:
pooled = stats.ttest_ind(music, silence, equal_var=True)

print(f't  = {pooled.statistic:.3f}')
print(f'df = {pooled.df:.0f}')
print(f'p  = {pooled.pvalue:.4f}')

t  = -2.576
df = 58
p  = 0.0126


## 4. Confidence interval for the difference

The test says whether the gap is real, and the interval says how big it might be. It is for music minus silence, so a fully negative interval means music scores are lower.

In [4]:
ci = welch.confidence_interval(confidence_level=0.95)
diff = music.mean() - silence.mean()

print(f'difference (music - silence) = {diff:.2f}')
print(f'95% CI: ({ci.low:.2f}, {ci.high:.2f})')
print('interval contains 0?', ci.low <= 0 <= ci.high)

difference (music - silence) = -5.00
95% CI: (-8.89, -1.11)
interval contains 0? False


## Try this

- Change the silence mean to 77 (edit the data by subtracting 2 from every silence score) and predict what happens to p.
- Cut each group to its first 10 scores (music[:10], silence[:10]) and predict whether the result stays significant.

## The same idea in R, SPSS and Stata

**R**

```r
music <- c(77.91, 82.44, 77.77, 62.24, 83.23, 78.87, 69.52, 80.15, 78.09, 77.42, 74.9, 79.82, 67.63, 73.08, 70.05, 80.32, 75.01, 71.85, 67.2, 72.18, 74.71, 72.01, 86.93, 84.2, 48.86, 56.67, 72.97, 70.62, 76.66, 76.69)
silence <- c(72.97, 64.46, 82.5, 83.47, 76.26, 80.89, 90.07, 85.63, 83.21, 83.08, 72.47, 84.97, 67.1, 76.11, 75.39, 66.56, 77.62, 89.61, 76.05, 70.12, 77.75, 84.22, 79.7, 78.36, 68.74, 81.8, 81.7, 87.55, 88.56, 83.08)

t.test(music, silence)                    # Welch (the R default)
t.test(music, silence, var.equal = TRUE)  # pooled
```

**SPSS** (Analyze > Compare Means > Independent-Samples T Test)

```
DATA LIST FREE / group score.
BEGIN DATA
1 77.91 1 82.44 1 77.77 1 62.24 1 83.23 1 78.87 1 69.52 1 80.15 1 78.09 1 77.42
1 74.9 1 79.82 1 67.63 1 73.08 1 70.05 1 80.32 1 75.01 1 71.85 1 67.2 1 72.18
1 74.71 1 72.01 1 86.93 1 84.2 1 48.86 1 56.67 1 72.97 1 70.62 1 76.66 1 76.69
2 72.97 2 64.46 2 82.5 2 83.47 2 76.26 2 80.89 2 90.07 2 85.63 2 83.21 2 83.08
2 72.47 2 84.97 2 67.1 2 76.11 2 75.39 2 66.56 2 77.62 2 89.61 2 76.05 2 70.12
2 77.75 2 84.22 2 79.7 2 78.36 2 68.74 2 81.8 2 81.7 2 87.55 2 88.56 2 83.08
END DATA.
T-TEST GROUPS=group(1 2)
  /VARIABLES=score
  /CRITERIA=CI(.95).
```

**Stata**

```
ttesti 30 74 8 30 79 7              // pooled (Stata's default)
ttesti 30 74 8 30 79 7, unequal     // Welch-type test, same df as R and Python
```